# Module 08 — Experiment Indexing and Metadata Reconstruction

Reconstruye, valida y cataloga ejecuciones experimentales a partir de stdout, stderr, seff y productos Darshan.

El producto principal es un **índice canónico por ejecución**. Las salidas de matriz/cobertura se mantienen por compatibilidad, pero se consideran salidas derivadas.


In [1]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

MODULE_FILE = Path("./Module/08_experiment_indexing.py").resolve()
spec = spec_from_file_location("deeptuneio_m08", MODULE_FILE)
m08 = module_from_spec(spec)
sys.modules[spec.name] = m08
spec.loader.exec_module(m08)
print("Module:", MODULE_FILE)
print("Version:", m08.VERSION)


Module: F:\NoteBooks\CORA_DR_UAB\Tesis\Scripts\DeepTuneIO\08_Experiment_Indexing\Module\08_experiment_indexing.py
Version: 1.0.1


In [2]:
# 1. Campaign selection
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": ["lustre_1ost", "lustre_2ost", "lustre_4ost", "nfs_SnGPU_e1"],
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"]},
        "NPZ_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"]},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
print(APPLICATION, DATASET_GROUP, SCENARIO)


DeepGalaxy DG_bw512_f3c5x64 nfs_SnGPU_e1


In [3]:
# 2. Routes — adapt only PROJECT_ROOT / ARTICLE_ID to the active article
PROJECT_ROOT = Path(r"D:\DL_Experiments\CORA_DR_UAB").resolve()
ARTICLE_ID = "Tesis"

ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID
CAMPAIGN_ROOT = ARTICLE_ROOT / "Data" / APPLICATION / DATASET_GROUP / SCENARIO
# M08 scans recursively. Point INPUT_DIR to the campaign directory containing the evidence files.
INPUT_DIR = CAMPAIGN_ROOT
OUTPUT_DIR = CAMPAIGN_ROOT / "results" / "Indexer"
RESOURCE_PROJECT = "RES-DATA-2022-1-0014"

print("Input :", INPUT_DIR)
print("Exists:", INPUT_DIR.exists())
print("Output:", OUTPUT_DIR)


Input : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Exists: True
Output: D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Indexer


In [5]:
# ==============================================================================
# 3. Execute M08
# ==============================================================================

required_vars = [
    "INPUT_DIR",
    "OUTPUT_DIR",
    "RESOURCE_PROJECT",
]

missing = [
    name for name in required_vars
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing notebook configuration: "
        + ", ".join(missing)
        + ". Execute the campaign and routes cells first."
    )

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Campaign input directory does not exist:\n{INPUT_DIR}"
    )

args = [
    str(INPUT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--resource-project", RESOURCE_PROJECT,
]

return_code = m08.main(args)

print("Return code:", return_code)

DeepTuneIO M08 — Experiment Indexing and Metadata Reconstruction v1.0.1
Input directory      : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Executions detected  : 26
Output directory     : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Indexer
csv                   : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Indexer\experiment_index_hdf5.csv
incomplete_csv        : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Indexer\incomplete_experiments_hdf5.csv
json                  : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Indexer\experiment_index_master_hdf5.json
validation_report     : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Indexer\validation_report_hdf5.txt
experiment_matrix     : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\Deep